# Experimento TCC — Correção Automatizada com e sem RAG

Este notebook executa o experimento completo do TCC via API do backend.

**Fluxo:**
1. Setup (registro, login, criação de turma e alunos)
2. Condição A — Com RAG (upload de PDF + publish)
3. Condição B — Sem RAG (publish sem PDF)
4. QA4 — Estabilidade (5 execuções da Condição A)
5. Coleta e exportação dos resultados

## 0. Configuração

In [42]:
import requests
import json
import time
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from pathlib import Path

BASE_URL = "http://localhost:8000"
PDF_PATH = Path("../notebooks/algoritmos-ufpr.pdf")
OUTPUT_DIR = Path("../tcc/results/v3")
OUTPUT_DIR.mkdir(exist_ok=True)

TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

assert PDF_PATH.exists(), f"PDF não encontrado em {PDF_PATH.resolve()}"

# Estado global do experimento
state = {}

def api(method, path, **kwargs):
    """Helper para chamadas à API com auth automático."""
    headers = kwargs.pop("headers", {})
    if "token" in state:
        headers["Authorization"] = f"Bearer {state['token']}"
    resp = getattr(requests, method)(f"{BASE_URL}{path}", headers=headers, **kwargs)
    if not resp.ok:
        print(f"❌ {method.upper()} {path} → {resp.status_code}")
        print(resp.text[:500])
    return resp

def save_csv(df, name):
    """Salva DataFrame como CSV em OUTPUT_DIR."""
    path = OUTPUT_DIR / f"{name}_{TIMESTAMP}.csv"
    df.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"  💾 {path}")
    return path

print("✅ Configuração carregada")
print(f"   Backend: {BASE_URL}")
print(f"   PDF:     {PDF_PATH.resolve()}")
print(f"   Output:  {OUTPUT_DIR.resolve()}")
print(f"   Run ID:  {TIMESTAMP}")

✅ Configuração carregada
   Backend: http://localhost:8000
   PDF:     C:\Users\Maycon Mendes\OneDrive\Área de Trabalho\IFF\TCC\ai-grading-system\notebooks\algoritmos-ufpr.pdf
   Output:  C:\Users\Maycon Mendes\OneDrive\Área de Trabalho\IFF\TCC\ai-grading-system\tcc\results\v3
   Run ID:  20260328_025927


## 1. Dados do Experimento

Questões, critérios e respostas

In [43]:
# ── Questões ──────────────────────────────────────────────────────────────────

QUESTIONS = [
    {
        "id": "Q1",
        "order": 1,
        "points": 10.0,
        "statement": (
            "Diferencie uma `Array` (vetor estático) de uma `Lista Encadeada Simples` "
            "quanto aos seguintes aspectos: alocação de memória, acesso a um elemento "
            "em uma posição arbitrária (índice `k`), e impacto da inserção/remoção de "
            "um elemento no meio da estrutura. Justifique suas respostas com exemplos "
            "de complexidade de tempo O(n)."
        ),
    },
    {
        "id": "Q2",
        "order": 2,
        "points": 10.0,
        "statement": (
            "Compare os algoritmos de ordenação `Merge Sort` e `Quick Sort` considerando "
            "os seguintes critérios: estabilidade, complexidade de tempo no pior caso e "
            "complexidade de espaço. Explique as implicações de cada característica para "
            "a escolha de um algoritmo em cenários específicos."
        ),
    },
    {
        "id": "Q3",
        "order": 3,
        "points": 10.0,
        "statement": (
            "Descreva as propriedades que definem uma `Árvore Binária de Busca (BST)` "
            "balanceada e uma não balanceada. Explique qual a principal vantagem de uma "
            "BST balanceada em relação a uma não balanceada, especificando a complexidade "
            "de tempo de busca, inserção e remoção em ambos os cenários (melhor e pior caso)."
        ),
    },
]

# ── Critérios de correção ─────────────────────────────────────────────────────

CRITERIAS = [
    {
        "uuid": "7457bc82-bbac-4355-ad74-637eeef93da0",
        "name": "Completude da resposta",
        "description": "Avalia se todos os pontos solicitados foram abordados de forma adequada.",
        "max_points": 10.0,
        "weight": 0.3
    },
    {
        "uuid": "808ac6a2-e24c-440a-9f07-613ada92d42c",
        "name": "Clareza e objetividade",
        "description": "Avalia se a resposta é clara, objetiva e fácil de compreender.",
        "max_points": 10.0,
        "weight": 0.2
    },
    {
        "uuid": "9dbc9e63-3b2d-4e0a-ad98-ec05d355bf61",
        "name": "Argumentação e justificativa",
        "description": "Avalia a qualidade das justificativas, fundamentação e consistência dos argumentos.",
        "max_points": 10.0,
        "weight": 0.1
    },
    {
        "uuid": "e91646ca-e9fc-43ea-8579-9f195567bb95",
        "name": "Precisão terminológica",
        "description": "Avalia o uso correto de termos técnicos e definições, evitando ambiguidade ou imprecisão.",
        "max_points": 10.0,
        "weight": 0.4
    }
]

# ── Alunos ────────────────────────────────────────────────────────────────────

STUDENTS = [
    {"name": "Aluno 1 (Excelente)",    "email": "aluno1.excelente@example.com",  "profile": "Excelente"},
    {"name": "Aluno 2 (Adequado)",     "email": "aluno2.adequado@example.com",   "profile": "Intermediária"},
    {"name": "Aluno 3 (Fraco)",        "email": "aluno3.fraco@example.com",      "profile": "Fraca"},
    {"name": "Aluno 4 (Fora do Tema)", "email": "aluno4.foratema@example.com",   "profile": "Fora tema"},
]

# Mapeamentos reutilizáveis
PROFILE_MAP = {s["name"]: s["profile"] for s in STUDENTS}
QUESTION_ORDER_TO_ID = {q["order"]: q["id"] for q in QUESTIONS}

# ── Respostas (12 = 3 questões × 4 alunos) ──────────────────────────────────
# Chave: (question_order, student_index)

ANSWERS = {
    # ── Q1: Array vs Lista Encadeada ──
    (1, 0): (
        "Arrays alocam memória contígua; Listas Encadeadas alocam nós não contiguamente. "
        "O acesso a um elemento `k` em Array é O(1) (direto por índice); em Lista é O(n), "
        "pois exige travessia sequencial. A inserção/remoção no meio em Array é O(n), devido "
        "ao deslocamento de elementos. Em Lista Encadeada, também é O(n) para localizar a "
        "posição e ajustar ponteiros."
    ),
    (1, 1): (
        "Uma Array aloca memória contígua, permitindo acesso direto a um elemento `k`. Já a "
        "Lista Encadeada Simples aloca nós dispersos, sendo o acesso a `k` feito por travessia, "
        "resultando em complexidade O(n). Inserir ou remover no meio de uma Array exige deslocar "
        "elementos, o que também custa O(n). Na Lista Encadeada, após achar o ponto, a "
        "inserção/remoção exige ajustar os ponteiros, podendo também levar tempo O(n) para localizar."
    ),
    (1, 2): (
        "`Arrays` alocam memória de forma estática em segmentos contíguos, facilitando o acesso "
        "O(1) a qualquer elemento via índice `k` por meio de um cálculo direto. Já `Listas "
        "Encadeadas` utilizam alocação dinâmica, onde cada nó armazena dados e ponteiros, o que "
        "exige um acesso O(n) para encontrar um elemento em `k`. A inserção/remoção no meio da "
        "`Array` causa um grande impacto de O(n) devido à necessidade de reorganização de todos "
        "os elementos, enquanto na `Lista` é mais eficiente, pois apenas ponteiros são alterados, "
        "mas a busca por esse ponto ainda é O(n)."
    ),
    (1, 3): (
        "A diferenciação entre planetas rochosos e gigantes gasosos reside na sua composição "
        "interna e formação. Enquanto rochosos possuem núcleo sólido e superfícies definidas, "
        "como a Terra, gigantes gasosos, como Júpiter, são vastas esferas de hidrogênio e hélio, "
        "sem superfície sólida clara. A observação de exoplanetas pode variar, revelando atmosferas "
        "ou a ausência delas, impactando a busca por vida."
    ),

    # ── Q2: Merge Sort vs Quick Sort ──
    (2, 0): (
        "Merge Sort (MS) é estável, O(n log n) tempo no pior caso e O(n) espaço auxiliar, ideal "
        "para estabilidade e desempenho garantido.\n"
        "Quick Sort (QS) é instável, O(n²) tempo no pior caso e O(log n) espaço médio (pilha "
        "recursiva), favorecendo eficiência de espaço.\n"
        "A estabilidade do MS é crucial para manter a ordem relativa de elementos iguais, e sua "
        "complexidade de tempo garantida é vital em sistemas críticos.\n"
        "O QS, embora mais rápido na prática para muitos cenários, exige boa escolha de pivô "
        "para evitar seu pior caso de tempo."
    ),
    (2, 1): (
        "Merge Sort é tipicamente estável e tem um desempenho de tempo mais previsível, sendo "
        "O(n log n) no pior caso, mas utiliza mais espaço auxiliar (O(n)). Quick Sort geralmente "
        "não é estável e pode ter um pior caso de tempo de O(n^2).\n\n"
        "No entanto, Quick Sort costuma ser mais eficiente em termos de espaço (O(log n) na "
        "média). Assim, escolhe-se Merge Sort quando a ordem relativa de elementos iguais é "
        "crucial ou a previsibilidade do tempo é prioritária.\n\n"
        "Quick Sort é preferível para menor uso de memória, mas é preciso considerar a variação "
        "do seu desempenho dependendo dos dados de entrada."
    ),
    (2, 2): (
        "Merge Sort é instável e tem complexidade de tempo de O(N^2) no pior caso, gastando "
        "pouco espaço. O Quick Sort é estável e mais eficiente, com O(N log N) no pior caso, "
        "mas exige mais memória. A escolha depende se a performance ou a estabilidade é a "
        "prioridade, sendo o Quick Sort melhor para dados já ordenados."
    ),
    (2, 3): (
        "Grelhar e assar diferem muito. Grelhar oferece sabor defumado intenso e cozimento "
        "rápido para carnes finas, mas requer atenção constante. Assar distribui calor "
        "uniformemente, ideal para bolos e pães, garantindo maciez interna. A complexidade "
        "do preparo e os equipamentos variam. A escolha impacta diretamente a textura final "
        "e a suculência do prato."
    ),

    # ── Q3: Árvore Binária de Busca ──
    (3, 0): (
        "Uma BST balanceada garante altura O(log n) (diferença de altura entre subárvores "
        "limitada); a não balanceada pode degenerar para O(n). A principal vantagem da "
        "balanceada é manter busca, inserção e remoção em O(log n) no melhor e pior caso. "
        "Já na não balanceada, estas operações são O(log n) no melhor caso, mas O(n) no "
        "pior caso (árvore degenerada), desempenho que a balanceada evita."
    ),
    (3, 1): (
        "Árvores balanceadas mantêm uma altura proporcional ao logaritmo do número de "
        "elementos, assegurando desempenho consistente O(log n) para busca, inserção e "
        "remoção no melhor e pior caso. Já as não balanceadas podem ter altura linear, "
        "degenerando para um pior caso O(n) nessas operações. A principal vantagem das "
        "balanceadas é a garantia de eficiência, evitando o desempenho imprevisível do "
        "pior caso linear das não balanceadas."
    ),
    (3, 2): (
        "Uma BST balanceada tem seus nós organizados de forma que todos os caminhos sejam "
        "de igual comprimento e cada nó filho é sempre maior que o pai, o que a torna "
        "otimizada. Uma não balanceada não segue essa regra, podendo virar uma lista. A "
        "vantagem principal da balanceada é sua agilidade. A complexidade de busca, inserção "
        "e remoção é O(log n) para a balanceada (todos os casos), e para a não balanceada, "
        "é O(1) no melhor e O(n!) no pior."
    ),
    (3, 3): (
        "Um pão balanceado requer levedura, farinha e água em proporções exatas para uma "
        "massa leve e aerada. Um pão não balanceado resulta em um miolo denso ou seco. A "
        "principal vantagem do balanceado é a melhor fermentação e sabor; o tempo de sova "
        "é O(1) no melhor caso, ou O(N) para ajustes no pior."
    ),
}

print(f"✅ Dados carregados: {len(QUESTIONS)} questões, {len(CRITERIAS)} critérios, {len(STUDENTS)} alunos, {len(ANSWERS)} respostas")

✅ Dados carregados: 3 questões, 4 critérios, 4 alunos, 12 respostas


## 2. Setup — Registro, Login, Turma e Alunos

### 2.1 Registro do professor

In [45]:
TEACHER = {
    "first_name": "Professor",
    "last_name": "TCC",
    "email": f"professor.tcc.{int(time.time())}@example.com",
    "password": "SenhaSegura@123",
    "user_type": "teacher",
}

r = api("post", "/users/register", json=TEACHER)
print(f"POST /users/register → {r.status_code}")

try:
    teacher_data = r.json()
except Exception:
    print("Resposta não-JSON:", r.text[:1000])
    r.raise_for_status()
if r.status_code >= 400:
    print("Resposta de erro:", r.status_code, r.text[:1000])
    r.raise_for_status()
# Extrair uuid de formatos comuns: top-level 'uuid' ou 'data' -> 'uuid'
teacher_uuid = teacher_data.get("uuid") or (teacher_data.get("data") or {}).get("uuid")
if not teacher_uuid:
    print("Resposta JSON (completa):")
    print(json.dumps(teacher_data, ensure_ascii=False, indent=2))
    raise KeyError("campo 'uuid' ausente na resposta do registro.")
state["teacher_uuid"] = teacher_uuid
print(f"✅ Professor registrado: {state['teacher_uuid']}")

POST /users/register → 201
✅ Professor registrado: 7792ddd4-73a8-40d1-81f2-d783189b7951


### 2.2 Login

In [46]:
r = api("post", "/auth/login", json={
    "email": TEACHER["email"],
    "password": TEACHER["password"],
})
r.raise_for_status()
state["token"] = r.json()["access_token"]
print(f"✅ Login OK — token obtido")

✅ Login OK — token obtido


### 2.3 Criar turma

In [47]:
r = api("post", "/classes", json={
    "name": "Algoritmos e Estrutura de Dados — TCC Experimento",
    "description": "Turma experimental para validação do sistema de correção automática",
    "year": 2026,
    "semester": 1,
})
r.raise_for_status()
state["class_uuid"] = r.json()["uuid"]
print(f"✅ Turma criada: {state['class_uuid']}")

✅ Turma criada: c0a7d477-3631-4eb7-94b8-2323df5b545b


### 2.4 Adicionar alunos

In [48]:
r = api("post", f"/classes/{state['class_uuid']}/students", json={
    "students": [
        {"full_name": s["name"], "email": s["email"]}
        for s in STUDENTS
    ]
})
print(f"POST /classes/.../students → {r.status_code}")
try:
    students_resp = r.json()
    print("Resposta JSON (students_resp):")
    print(json.dumps(students_resp, ensure_ascii=False, indent=2)[:2000])
except Exception:
    print("Resposta não-JSON:", r.text[:1000])
    r.raise_for_status()
if r.status_code >= 400:
    print("Resposta de erro:", r.status_code, r.text[:1000])
    r.raise_for_status()
# Extrair alunos de formatos possíveis: top-level 'students'|'enrolled' ou 'details'
details = students_resp.get("details", {}) if isinstance(students_resp, dict) else {}
enrolled = students_resp.get("students") or students_resp.get("enrolled") or details.get("enrolled") or []
created = details.get("created", []) or []
# Combinar mantendo ordem e evitando duplicatas simples
all_new = list(enrolled) + [c for c in created if c not in enrolled]
if not isinstance(all_new, list):
    print("Formato inesperado na resposta (esperado lista):")
    print(json.dumps(students_resp, ensure_ascii=False, indent=2))
    raise KeyError("Lista de alunos não encontrada na resposta")
# Extrair uuids tolerando 'uuid' ou 'id'
state["student_uuids"] = [s.get("uuid") or s.get("id") for s in all_new]
for i, s in enumerate(all_new):
    uuid = s.get("uuid") or s.get("id") or "<sem uuid>"
    profile = STUDENTS[i]['profile'] if i < len(STUDENTS) else '<desconhecido>'
    print(f"  {profile:>10} → {uuid}")
print(f"✅ {len(all_new)} alunos adicionados")
# Salvar CSV com alunos criados / matriculados
try:
    df_students = pd.DataFrame(all_new)
    if not df_students.empty:
        cols = [c for c in ["uuid", "id", "full_name", "email"] if c in df_students.columns]
        df_students = df_students[cols] if cols else df_students
        save_csv(df_students, "students_added")
except Exception as e:
    print("Erro ao salvar CSV de alunos:", e)

POST /classes/.../students → 200
Resposta JSON (students_resp):
{
  "class_uuid": "c0a7d477-3631-4eb7-94b8-2323df5b545b",
  "class_name": "Algoritmos e Estrutura de Dados — TCC Experimento",
  "summary": {
    "total_requested": 4,
    "students_enrolled": 4,
    "students_already_enrolled": 0,
    "new_students_created": 0,
    "errors": 0
  },
  "details": {
    "enrolled": [
      {
        "uuid": "6123ae8c-1aca-43c5-96c3-57b4397e8dfc",
        "full_name": "Aluno 1 (Excelente)",
        "email": "aluno1.excelente@example.com"
      },
      {
        "uuid": "37a931aa-52de-4e5f-a36e-d463800f0e2e",
        "full_name": "Aluno 2 (Adequado)",
        "email": "aluno2.adequado@example.com"
      },
      {
        "uuid": "875740fd-89c3-4acd-bc4e-a9fd36acacc0",
        "full_name": "Aluno 3 (Fraco)",
        "email": "aluno3.fraco@example.com"
      },
      {
        "uuid": "c49ec29a-0367-46c0-8760-763506c24f08",
        "full_name": "Aluno 4 (Fora do Tema)",
        "email": "aluno

## DEF run_experiment

In [49]:
def run_experiment(label, upload_pdf=True, poll_interval=5, max_wait=300):    
    
    ts = datetime.now().strftime("%H:%M:%S")
    print(f"\n{'='*70}")
    print(f"  {label} — início {ts}")
    print(f"  RAG: {'SIM' if upload_pdf else 'NÃO'}")
    print(f"{'='*70}\n")

    # ── Criar prova ───────────────────────────────────────────────────────
    now = datetime.now()
    r = api("post", "/exams", json={
        "title": f"{label} — {now.strftime('%Y%m%d_%H%M%S')}",
        "description": f"Experimento TCC: {label}",
        "class_uuid": state["class_uuid"],
        "starts_at": now.isoformat(),
        "ends_at": (now + timedelta(hours=2)).isoformat(),
    })
    r.raise_for_status()
    exam_uuid = r.json()["uuid"]
    print(f"  📝 Prova criada: {exam_uuid}")

    # ── Criar critérios de avaliação (exam-criteria) a partir de CRITERIAS ──
    try:
        criteria_added = 0
        for c in CRITERIAS:
            try:
                r2 = api("post", "/exam-criteria", json={
                    "exam_uuid": exam_uuid,
                    "criteria_uuid": c.get("uuid"),
                    "weight": c.get("weight", 1.0),
                    "max_points": c.get("max_points", 10.0),
                })
                r2.raise_for_status()
                criteria_added += 1
            except Exception:
                pass
        print(f"  ✅ {criteria_added} critérios configurados")
    except Exception as e:
        print("⚠️ Erro ao configurar critérios:", e)

    # ── Criar questões ────────────────────────────────────────────────────
    question_uuids = []
    for q in QUESTIONS:
        r = api("post", "/exam-questions", json={
            "exam_uuid": exam_uuid,
            "statement": q["statement"],
            "question_order": q["order"],
            "points": q["points"],
        })
        r.raise_for_status()
        question_uuids.append(r.json()["uuid"])
    print(f"  📋 {len(question_uuids)} questões criadas")

    # Mapear UUID da questão criada → Q-id (Q1, Q2, Q3)
    uuid_to_qid = {question_uuids[i]: QUESTIONS[i]["id"] for i in range(len(QUESTIONS))}

    # ── Submeter respostas ────────────────────────────────────────────────
    answer_count = 0
    for (q_order, s_idx), answer_text in ANSWERS.items():
        r = api("post", "/student-answers", json={
            "exam_uuid": exam_uuid,
            "question_uuid": question_uuids[q_order - 1],
            "student_uuid": state["student_uuids"][s_idx],
            "answer_text": answer_text,
        })
        r.raise_for_status()
        answer_count += 1
    print(f"  ✏️  {answer_count} respostas submetidas")

    # ── Upload PDF (RAG) ──────────────────────────────────────────────────
    if upload_pdf:
        with open(PDF_PATH, "rb") as f:
            r = api("post", f"/attachments/upload",
                    params={"exam_uuid": exam_uuid},
                    files={"file": (PDF_PATH.name, f, "application/pdf")})
        r.raise_for_status()
        print(f"  📄 PDF enviado para RAG")
    else:
        print(f"  📄 Sem PDF — condição sem RAG")

    # ── Publicar (dispara correção em background) ─────────────────────────
    r = api("post", f"/exams/{exam_uuid}/publish")
    r.raise_for_status()
    print(f"  🚀 Prova publicada — correção iniciada")

    # ── Polling até GRADED ────────────────────────────────────────────────
    start = time.time()
    while time.time() - start < max_wait:
        time.sleep(poll_interval)
        r = api("get", f"/exams/{exam_uuid}")
        r.raise_for_status()
        status = r.json().get("status", "")
        elapsed = int(time.time() - start)
        print(f"  ⏳ {elapsed}s — status: {status}        ", end="\r")
        if status in ("GRADED", "WARNING"):
            print(f"  ✅ Correção finalizada em {elapsed}s — status: {status}")
            break
    else:
        print(f"  ⚠️ Timeout ({max_wait}s) — verificar manualmente")

    # ── Coletar resultados via review ─────────────────────────────────────
    r = api("get", f"/reviews/exams/{exam_uuid}")
    r.raise_for_status()
    review = r.json()

    results = []
    for q_data in review.get("questions", []):
        q_uuid = str(q_data.get("question_uuid", q_data.get("uuid", "")))
        q_id = uuid_to_qid.get(q_uuid, "?")

        for ans in q_data.get("student_answers", q_data.get("answers", [])):
            results.append({
                "exam_uuid": exam_uuid,
                "label": label,
                "rag": upload_pdf,
                "question_id": q_id,
                "question": q_data.get("statement", "")[:60],
                "question_uuid": q_uuid,
                "student_name": ans.get("student_name", ""),
                "profile": PROFILE_MAP.get(ans.get("student_name", ""), "?"),
                "student_uuid": ans.get("student_uuid"),
                "score": ans.get("score"),
                "c1_score": ans.get("c1_score"),
                "c2_score": ans.get("c2_score"),
                "arbiter_score": ans.get("arbiter_score"),
                "divergence_detected": ans.get("divergence_detected", False),
                "divergence_value": ans.get("divergence_value"),
                "consensus_method": ans.get("consensus_method"),
                "feedback": ans.get("feedback", ""),
            })

    df = pd.DataFrame(results)
    print(f"\n  📊 {len(results)} resultados coletados")
    if len(df) > 0:
        print(f"  📈 Média geral: {df['score'].mean():.2f}")
        n_div = df["divergence_detected"].sum()
        if n_div > 0:
            print(f"  ⚠️  {n_div} divergências detectadas (árbitro acionado)")

    return {
        "label": label,
        "exam_uuid": exam_uuid,
        "rag": upload_pdf,
        "df": df,
        "raw_review": review,
    }

## 4. Condição A — Com RAG

In [50]:
cond_a = run_experiment("Condição A (com RAG)", upload_pdf=True)


  Condição A (com RAG) — início 03:02:22
  RAG: SIM

  📝 Prova criada: 0db5d257-3d1e-48d2-9ca4-a11db57f0594
  ✅ 4 critérios configurados
  📋 3 questões criadas
  ✏️  12 respostas submetidas
  📄 PDF enviado para RAG
  🚀 Prova publicada — correção iniciada
  ✅ Correção finalizada em 155s — status: GRADED

  📊 12 resultados coletados
  📈 Média geral: 6.23


In [ ]:
# Tabela detalhada + salva CSV
cols = ["question_id", "profile", "c1_score", "c2_score", "arbiter_score", "score", "divergence_detected", "consensus_method"]
display(cond_a["df"][cols])

save_csv(cond_a["df"], "cond_A")

## 5. Condição B — Sem RAG

In [ ]:
cond_b = run_experiment("Condição B (sem RAG)", upload_pdf=False)

In [ ]:
# Tabela detalhada + salva CSV
cols = ["question_id", "profile", "c1_score", "c2_score", "arbiter_score", "score", "divergence_detected", "consensus_method"]
display(cond_b["df"][cols])

save_csv(cond_b["df"], "cond_B")

## 6. Comparação A vs B

In [ ]:
# Comparação pareada A vs B com detalhes dos corretores
a = cond_a["df"][["question_id", "profile", "c1_score", "c2_score", "arbiter_score", "score", "divergence_detected"]].copy()
b = cond_b["df"][["question_id", "profile", "c1_score", "c2_score", "arbiter_score", "score", "divergence_detected"]].copy()

a.columns = ["question_id", "profile", "A_c1", "A_c2", "A_arb", "A_final", "A_div"]
b.columns = ["question_id", "profile", "B_c1", "B_c2", "B_arb", "B_final", "B_div"]

comparison = a.merge(b, on=["question_id", "profile"])
comparison["Delta"] = comparison["A_final"] - comparison["B_final"]

display(comparison)

save_csv(comparison, "comparacao_A_vs_B")

In [ ]:
# Resumo por nível de qualidade
summary = comparison.groupby("profile").agg(
    A_final=("A_final", "mean"),
    B_final=("B_final", "mean"),
    Delta=("Delta", "mean"),
).round(2)

print("Média por nível de qualidade:")
display(summary)

save_csv(summary.reset_index(), "resumo_por_nivel")

## 7. QA4 — Estabilidade (5 execuções com RAG)

In [ ]:
QA4_RUNS = 5
qa4_results = []

for i in range(QA4_RUNS):
    result = run_experiment(f"QA4 Run {i+1}/{QA4_RUNS}", upload_pdf=True)
    result["df"]["run"] = i + 1
    qa4_results.append(result)
    print(f"\n{'─'*70}\n")

print(f"\n✅ QA4 completo — {QA4_RUNS} execuções realizadas")

In [ ]:
# Consolidar todas as runs do QA4
all_qa4 = pd.concat([r["df"] for r in qa4_results], ignore_index=True)

# Salva CSV com todas as runs brutas (cada linha = 1 correção)
save_csv(all_qa4, "qa4_todas_runs")

# Estatísticas por aluno/questão
qa4_stats = all_qa4.groupby(["question_id", "profile"]).agg(
    mean_score=("score", "mean"),
    std_score=("score", "std"),
    min_score=("score", "min"),
    max_score=("score", "max"),
    var_score=("score", "var"),
    mean_c1=("c1_score", "mean"),
    mean_c2=("c2_score", "mean"),
    n_divergencias=("divergence_detected", "sum"),
).round(2)

print(f"Variância média: {qa4_stats['var_score'].mean():.2f}")
print(f"Desvio padrão médio: {qa4_stats['std_score'].mean():.2f}")
print()
display(qa4_stats)

save_csv(qa4_stats.reset_index(), "qa4_estatisticas")

In [ ]:
# Resumo QA4 por nível de qualidade
qa4_by_profile = all_qa4.groupby("profile").agg(
    mean_score=("score", "mean"),
    std_score=("score", "std"),
    var_score=("score", "var"),
    mean_c1=("c1_score", "mean"),
    mean_c2=("c2_score", "mean"),
    n_divergencias=("divergence_detected", "sum"),
).round(2)

print("QA4 — Estabilidade por nível:")
display(qa4_by_profile)

save_csv(qa4_by_profile.reset_index(), "qa4_resumo_por_nivel")

## 8. Exportar JSON + Resumo Final

In [ ]:
# Salvar raw reviews completos em JSON (backup rastreável)
for run_data, name in [(cond_a, "cond_A"), (cond_b, "cond_B")]:
    path = OUTPUT_DIR / f"{name}_{TIMESTAMP}.json"
    with open(path, "w", encoding="utf-8") as f:
        json.dump({
            "label": run_data["label"],
            "exam_uuid": run_data["exam_uuid"],
            "rag": run_data["rag"],
            "timestamp": TIMESTAMP,
            "raw_review": run_data["raw_review"],
        }, f, ensure_ascii=False, indent=2, default=str)
    print(f"  💾 {path}")

# QA4 JSON
qa4_path = OUTPUT_DIR / f"qa4_{TIMESTAMP}.json"
with open(qa4_path, "w", encoding="utf-8") as f:
    json.dump({
        "label": "QA4 Estabilidade",
        "num_runs": QA4_RUNS,
        "timestamp": TIMESTAMP,
        "runs": [
            {"run": i + 1, "exam_uuid": r["exam_uuid"], "raw_review": r["raw_review"]}
            for i, r in enumerate(qa4_results)
        ],
    }, f, ensure_ascii=False, indent=2, default=str)
print(f"  💾 {qa4_path}")

In [ ]:
# Resumo final
print("="*70)
print("  RESUMO DO EXPERIMENTO")
print("="*70)
print(f"\n  Condição A (com RAG):  média = {cond_a['df']['score'].mean():.2f}")
print(f"  Condição B (sem RAG):  média = {cond_b['df']['score'].mean():.2f}")
print(f"  Delta médio (A - B):   {comparison['Delta'].mean():+.2f}")
print(f"\n  QA4 — {QA4_RUNS} execuções:")
print(f"    Variância média:     {qa4_stats['var_score'].mean():.2f}")
print(f"    Desvio padrão médio: {qa4_stats['std_score'].mean():.2f}")

print(f"\n  CSVs salvos em: {OUTPUT_DIR.resolve()}/")
print(f"    cond_A_{TIMESTAMP}.csv")
print(f"    cond_B_{TIMESTAMP}.csv")
print(f"    comparacao_A_vs_B_{TIMESTAMP}.csv")
print(f"    resumo_por_nivel_{TIMESTAMP}.csv")
print(f"    qa4_todas_runs_{TIMESTAMP}.csv")
print(f"    qa4_estatisticas_{TIMESTAMP}.csv")
print(f"    qa4_resumo_por_nivel_{TIMESTAMP}.csv")
print("="*70)